# GitHub Actions for Data Engineering — dbt CI, GE, Matrix Testing

## Mental Model

Think of a data CI pipeline like an airport security gate for analytics code.

- **dbt compile/test** checks whether transformation logic still fits the warehouse contract.
- **Great Expectations** checks whether the data itself still behaves like the business expects.
- **Matrix testing** checks whether the same workflow survives across multiple runtime combinations without duplicating YAML.

For a Senior Data Engineer, GitHub Actions becomes the **automated reviewer** that runs on every PR before code is merged.

### Scenario Context

This notebook is grounded in a realistic telemetry platform:

- PostgreSQL database: `de_telemetry`
- `endpoints`: 10,000 rows
- `metrics`: 500,000 rows
- `alerts`: 25,000 rows
- Operational narrative: 6,000+ API endpoints monitored for latency, error rate, and throughput, with alerts escalating through severity tiers.

The goal is to build CI/CD workflow assets that validate this pipeline locally and save production-style workflow YAML files into a GitHub repo structure under:

`D:/Workspace/Technologies/.github/workflows/`


In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import textwrap
from pathlib import Path

try:
    import psycopg2
except ImportError as exc:
    raise RuntimeError("psycopg2 is required for this notebook.") from exc

DBT_PATH = Path(r"C:/py_venv/proj_educate/Scripts/dbt.exe")
WORKSPACE_ROOT = Path(r"D:/Workspace/Technologies")
WORKFLOWS_DIR = WORKSPACE_ROOT / ".github" / "workflows"
WORKFLOWS_DIR.mkdir(parents=True, exist_ok=True)

PG_CONN = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

TECH_STACK = {
    "kafka": {"bootstrap_servers": "localhost:9092", "container": "citi_kafka", "image": "confluentinc/cp-kafka:7.6.0"},
    "spark": {"version": "pyspark==3.5.4", "master": "local[*]", "JAVA_HOME": r"C:/Program Files/Java/jre1.8.0_481", "HADOOP_HOME": r"C:/hadoop"},
    "airflow": {"url": "http://localhost:8082", "version": "apache/airflow:2.8.0", "executor": "LocalExecutor", "username": "admin", "password": "admin"},
    "mlflow": {"url": "http://localhost:5000", "backend": "sqlite"},
    "dbt": {"path": str(DBT_PATH), "project": "citi_dbt", "target": "postgres", "profiles": str(Path.home() / ".dbt" / "profiles.yml")},
    "databricks": {"host": "https://dbc-9f35a83d-b4e7.cloud.databricks.com", "warehouse_id": "b6657f31d1e7a179"},
    "gcp": {"project": "citi-de-learning", "key": r"D:/Workspace/Technologies/_setup/gcp_key.json"},
    "azure": {"subscription": "b3811436-61fc-4a3a-a6a9-deb05955076d", "az_cli": r"C:\Program Files (x86)\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"},
    "aws": {"profile": "study", "region": "us-east-1", "account": "357811130281"},
}

def run_command(cmd, cwd=None, check=True, env=None):
    """Run a subprocess and return CompletedProcess while printing useful diagnostics."""
    printable = cmd if isinstance(cmd, str) else " ".join(str(part) for part in cmd)
    print(f"\n$ {printable}")
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        capture_output=True,
        shell=isinstance(cmd, str),
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {printable}")
    return result

def write_text_file(path: Path, content: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8", newline="\n")
    print(f"Saved: {path}")
    return path

def postgres_scalar(sql: str):
    with psycopg2.connect(**PG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            row = cur.fetchone()
            return row[0] if row else None

def validate_dataset_context():
    checks = {
        "endpoints_rows": "select count(*) from endpoints;",
        "metrics_rows": "select count(*) from metrics;",
        "alerts_rows": "select count(*) from alerts;",
        "endpoint_pk_nulls": "select count(*) from endpoints where endpoint_id is null;",
        "metric_fk_nulls": "select count(*) from metrics where endpoint_id is null;",
        "alerts_fk_nulls": "select count(*) from alerts where endpoint_id is null;",
    }
    results = {name: postgres_scalar(sql) for name, sql in checks.items()}
    print(json.dumps(results, indent=2))
    return results

dataset_results = validate_dataset_context()

assert dataset_results["endpoints_rows"] == 10000, f"Expected 10,000 endpoints, got {dataset_results['endpoints_rows']}"
assert dataset_results["metrics_rows"] == 500000, f"Expected 500,000 metrics, got {dataset_results['metrics_rows']}"
assert dataset_results["alerts_rows"] == 25000, f"Expected 25,000 alerts, got {dataset_results['alerts_rows']}"
assert dataset_results["endpoint_pk_nulls"] == 0, "Found null endpoint_id values in endpoints."
assert dataset_results["metric_fk_nulls"] == 0, "Found null endpoint_id values in metrics."
assert dataset_results["alerts_fk_nulls"] == 0, "Found null endpoint_id values in alerts."

print("Environment and dataset validation passed.")


## Setup

A production notebook should be explicit about what it assumes.

- We use `subprocess` for dbt because CI runners invoke dbt from the command line.
- We do **not** install packages inside the notebook.
- We validate the local PostgreSQL dataset first so the rest of the CI story is tied to a real warehouse state.
- We save workflow files directly into `.github/workflows` so the notebook acts like a workflow generator.


In [2]:
dbt_ci_yml = textwrap.dedent(
    """
    name: dbt CI

    on:
      pull_request:
        branches:
          - main
          - master

    jobs:
      dbt-ci:
        runs-on: ubuntu-latest
        env:
          DBT_PROFILES_DIR: ~/.dbt
        steps:
          - name: Checkout repository
            uses: actions/checkout@v4

          - name: Set up Python
            uses: actions/setup-python@v5
            with:
              python-version: "3.11"

          - name: Install dbt
            run: |
              python -m pip install --upgrade pip
              pip install dbt-postgres

          - name: Restore dbt packages cache
            uses: actions/cache@v4
            with:
              path: |
                ~/.dbt
                dbt_packages
              key: ${{ runner.os }}-dbt-${{ hashFiles('**/packages.yml', '**/package-lock.yml', '**/poetry.lock', '**/requirements*.txt') }}
              restore-keys: |
                ${{ runner.os }}-dbt-

          - name: dbt deps
            run: dbt deps --profiles-dir ~/.dbt --project-dir .

          - name: dbt compile
            run: dbt compile --profiles-dir ~/.dbt --project-dir .

          - name: dbt test modified models
            run: dbt test --profiles-dir ~/.dbt --project-dir . --select state:modified+
    """
).strip() + "\n"

dbt_ci_path = write_text_file(WORKFLOWS_DIR / "dbt-ci.yml", dbt_ci_yml)
print(dbt_ci_yml)


## dbt CI Workflow

This workflow is optimized for pull requests:

- It checks out code
- Installs dbt
- Resolves dependencies
- Compiles models
- Runs tests only for `state:modified+`

That selection pattern is important because it gives you **fast feedback on changed models and their downstream blast radius**, instead of rerunning the entire warehouse on every PR.


In [3]:
def locate_ge_project_root() -> Path:
    candidates = [
        WORKSPACE_ROOT / "cicd_data_intro",
        WORKSPACE_ROOT / "cicd_data_intro" / "great_expectations",
        Path.cwd(),
    ]

    for candidate in candidates:
        if candidate.is_dir():
            if (candidate / "great_expectations.yml").exists():
                return candidate
            if (candidate / "great_expectations").is_dir():
                return candidate

    for root in [WORKSPACE_ROOT, Path.cwd()]:
        for path in root.rglob("great_expectations.yml"):
            return path.parent

    raise FileNotFoundError("Could not locate a Great Expectations project rooted under cicd_data_intro or the current working directory.")

def ge_checkpoint_result_to_dict(result):
    if hasattr(result, "to_json_dict"):
        return result.to_json_dict()
    if hasattr(result, "to_dict"):
        return result.to_dict()
    if isinstance(result, dict):
        return result
    return {"repr": repr(result)}

def run_ge_checkpoint_fail_fast(checkpoint_name: str | None = None, output_path: Path | None = None):
    try:
        import great_expectations as gx
    except ImportError as exc:
        raise RuntimeError("great_expectations must already be installed for this notebook.") from exc

    project_root = locate_ge_project_root()
    context = gx.get_context(context_root_dir=str(project_root))

    checkpoint_names = []
    if hasattr(context, "checkpoints"):
        if hasattr(context.checkpoints, "all"):
            checkpoint_names = [cp.name for cp in context.checkpoints.all()]
        elif hasattr(context.checkpoints, "list"):
            checkpoint_names = [cp["name"] if isinstance(cp, dict) else cp.name for cp in context.checkpoints.list()]

    if not checkpoint_names and hasattr(context, "list_checkpoints"):
        checkpoint_names = [cp["name"] if isinstance(cp, dict) else cp.name for cp in context.list_checkpoints()]

    if checkpoint_name is None:
        if not checkpoint_names:
            raise RuntimeError(f"No Great Expectations checkpoints found in project: {project_root}")
        checkpoint_name = checkpoint_names[0]

    print(f"GE project root: {project_root}")
    print(f"Running checkpoint: {checkpoint_name}")

    checkpoint_result = context.run_checkpoint(checkpoint_name=checkpoint_name)
    result_dict = ge_checkpoint_result_to_dict(checkpoint_result)

    if output_path is None:
        output_path = WORKSPACE_ROOT / "ge_checkpoint_result.json"

    write_text_file(output_path, json.dumps(result_dict, indent=2, default=str))

    success = None
    if isinstance(result_dict, dict):
        if "success" in result_dict:
            success = bool(result_dict["success"])
        elif "checkpoint_result" in result_dict and isinstance(result_dict["checkpoint_result"], dict):
            success = bool(result_dict["checkpoint_result"].get("success"))

    if success is None and hasattr(checkpoint_result, "success"):
        success = bool(checkpoint_result.success)

    print(json.dumps({"checkpoint_name": checkpoint_name, "success": success}, indent=2))

    if success is not True:
        raise SystemExit(f"Checkpoint failed: {checkpoint_name}")

    return result_dict

ge_result = run_ge_checkpoint_fail_fast()
print("Great Expectations checkpoint passed. Fail-fast gate is armed and would terminate on any failed expectation.")


## GE Checkpoint in CI

This is the **data-quality gate**.

A schema test might tell you a table still exists.  
A Great Expectations checkpoint tells you whether the **data is still believable**.

Typical examples:

- unexpected null spikes
- invalid severity values
- out-of-range metrics
- referential breaks between `alerts` and `endpoints`

The notebook runs the checkpoint locally, writes the result JSON artifact, and stops the execution immediately if any expectation fails.


In [4]:
matrix_yml = textwrap.dedent(
    """
    name: Matrix Validation

    on:
      pull_request:
      workflow_dispatch:

    jobs:
      matrix-ci:
        runs-on: ubuntu-latest
        strategy:
          fail-fast: false
          matrix:
            python-version: ["3.10", "3.11", "3.12"]
            dbt-adapter: ["postgres", "snowflake"]

        steps:
          - name: Checkout repository
            uses: actions/checkout@v4

          - name: Set up Python
            uses: actions/setup-python@v5
            with:
              python-version: ${{ matrix.python-version }}

          - name: Install dbt adapter
            run: |
              python -m pip install --upgrade pip
              pip install dbt-${{ matrix.dbt-adapter }}

          - name: Print matrix combination
            run: |
              echo "Python version: ${{ matrix.python-version }}"
              echo "dbt adapter: ${{ matrix.dbt-adapter }}"

          - name: dbt compile
            run: dbt compile --profiles-dir ~/.dbt --project-dir .
    """
).strip() + "\n"

matrix_path = write_text_file(WORKFLOWS_DIR / "matrix.yml", matrix_yml)
print(matrix_yml)

print(
    "\nMatrix strategy explanation:\n"
    "- One YAML template expands into multiple jobs.\n"
    "- That removes repeated workflow blocks.\n"
    "- It validates compatibility across Python runtimes and adapters in parallel."
)


## Matrix Testing

Matrix testing is basically a **Cartesian product of trust**.

Instead of manually copying the same workflow three or six times, GitHub Actions expands one YAML definition into multiple jobs.

Here that means:

- Python: `3.10`, `3.11`, `3.12`
- dbt adapters: `postgres`, `snowflake`

So one workflow definition covers six runtime combinations without duplicated logic.


In [5]:
secrets_notes = {
    "github_secrets": [
        "Encrypted at the repository, environment, or organization level.",
        "Best for passwords, tokens, cloud keys, and connection strings.",
        "Referenced in workflows with ${{ secrets.SECRET_NAME }}."
    ],
    "environment_variables": [
        "Useful for non-secret runtime configuration.",
        "Examples: environment name, region, dbt target profile, feature flags.",
        "Should never hold plain-text credentials in committed workflow files."
    ],
    "rule": "Never hardcode credentials directly inside workflow YAML committed to source control."
}

secrets_example_yml = textwrap.dedent(
    """
    env:
      POSTGRES_HOST: localhost
      POSTGRES_DB: de_telemetry
      POSTGRES_USER: de_admin
      POSTGRES_PASSWORD: ${{ secrets.POSTGRES_PASSWORD }}

    steps:
      - name: Example secure connection usage
        run: |
          echo "Connecting to ${POSTGRES_HOST}/${POSTGRES_DB} as ${POSTGRES_USER}"
    """
).strip() + "\n"

print(json.dumps(secrets_notes, indent=2))
print()
print(secrets_example_yml)

assert "${{ secrets.POSTGRES_PASSWORD }}" in secrets_example_yml
print("Secrets example validated.")


## Secrets Management

A clean mental split:

- **GitHub Secrets** = sensitive values
- **Environment variables** = regular runtime configuration

Bad pattern:
- hardcoding passwords in workflow files

Good pattern:
- keep the host, db name, and non-secret config in `env`
- keep the password/token/key in `${{ secrets.* }}`

That keeps CI readable without leaking credentials into version control.


In [6]:
end_to_end_ci_yml = textwrap.dedent(
    """
    name: Data Pipeline CI Flow

    on:
      pull_request:
        branches:
          - main
          - master
      workflow_dispatch:

    jobs:
      data-ci:
        runs-on: ubuntu-latest
        env:
          DBT_PROFILES_DIR: ~/.dbt
          POSTGRES_HOST: localhost
          POSTGRES_DB: de_telemetry
          POSTGRES_USER: de_admin
          POSTGRES_PASSWORD: ${{ secrets.POSTGRES_PASSWORD }}

        steps:
          - name: Checkout repository
            uses: actions/checkout@v4

          - name: Set up Python
            uses: actions/setup-python@v5
            with:
              python-version: "3.11"

          - name: Install CI dependencies
            run: |
              python -m pip install --upgrade pip
              pip install dbt-postgres great_expectations sqlfluff

          - name: Lint SQL
            run: sqlfluff lint .

          - name: dbt deps
            run: dbt deps --profiles-dir ~/.dbt --project-dir .

          - name: dbt test
            run: dbt test --profiles-dir ~/.dbt --project-dir .

          - name: Great Expectations checkpoint
            run: |
              python - <<'PY'
              import great_expectations as gx
              from pathlib import Path

              project_root = Path("cicd_data_intro")
              if not project_root.exists():
                  project_root = Path(".")
              context = gx.get_context(context_root_dir=str(project_root))
              checkpoints = []
              if hasattr(context, "checkpoints") and hasattr(context.checkpoints, "all"):
                  checkpoints = [cp.name for cp in context.checkpoints.all()]
              elif hasattr(context, "list_checkpoints"):
                  checkpoints = [cp["name"] if isinstance(cp, dict) else cp.name for cp in context.list_checkpoints()]
              if not checkpoints:
                  raise SystemExit("No Great Expectations checkpoint found.")
              result = context.run_checkpoint(checkpoint_name=checkpoints[0])
              success = getattr(result, "success", None)
              if success is None and hasattr(result, "to_json_dict"):
                  success = result.to_json_dict().get("success")
              if success is not True:
                  raise SystemExit("Great Expectations checkpoint failed.")
              print("Great Expectations checkpoint passed.")
              PY

          - name: Deploy transformations
            if: github.event_name == 'workflow_dispatch' || github.ref == 'refs/heads/main' || github.ref == 'refs/heads/master'
            run: dbt run --profiles-dir ~/.dbt --project-dir .
    """
).strip() + "\n"

end_to_end_path = write_text_file(WORKFLOWS_DIR / "data-pipeline-ci.yml", end_to_end_ci_yml)
print(end_to_end_ci_yml)


## Data Pipeline CI Flow

This is the end-to-end quality gate:

1. **Lint SQL** with `sqlfluff`
2. **Run dbt tests**
3. **Run Great Expectations checkpoint**
4. **Deploy** with `dbt run`

That sequence turns CI into a **code contract** for data engineering.

Why that matters in the Citi-style telemetry story:

- schema drift gets caught before merge
- null explosions get caught before dashboards break
- referential integrity breaks get caught before downstream alerts misroute
- deploy only happens after quality gates pass


## What Just Happened

A CI pipeline for data is a **quality gate**.

Every PR runs dbt tests plus Great Expectations validations before merge.  
That means the team catches:

- schema drift
- null violations
- referential integrity breaks
- runtime compatibility issues

...before those issues hit production.

For a telemetry-heavy environment like Citi's, where 6,000+ API endpoints are monitored for latency, error rate, and throughput, that gate is not just convenience.

It is **operational risk control**.
